# Join Reordering
Spark changes the order in which joins are executed to minimise cost.
<br>
(A JOIN B) JOIN C<br>
vs<br>
(B JOIN C) JOIN A<br>
gives the same result, but a very different runtime cost

why?


# Pre-aggregation and Pre-filtering before join

Spark Optimiser Catalyst will:
1. Inline subqueries
2. Push filters down
3. Remove redundant projections

But Catalyst cannot invent pre-aggregation for you — you must write it.

```
orders.join(customers, "customer_id")
      .groupBy("country")
      .sum("amount")
```

### Problem:
Join happens on millions of rows
<br>Massive shuffle


### Solution:
Aggregate as early as possible, before joins. Joins become tiny

```
orders_agg = (
    orders
    .groupBy("customer_id")
    .agg(sum("amount").alias("total_amount"))
)

orders_agg.join(customers, "customer_id")
          .groupBy("country")
          .sum("total_amount")

```

More granularity 
```
orders
  .groupBy("customer_id", "date")
  .agg(sum("amount").alias("daily_amount"))
  .join(customers, "customer_id")
```

# Filter + Project Before Join

Below is an example of a bad query 

```
SELECT o.customer_id, SUM(o.amount), c.country
FROM orders o
JOIN customers c
  ON o.customer_id = c.customer_id
WHERE o.order_date >= '2025-01-10'
GROUP BY o.customer_id, c.country;
```

Good Query

```
SELECT o.customer_id, total_amount, c.country
FROM (
    SELECT customer_id, sum(amount) AS total_amount
    FROM orders
    WHERE order_date >= '2025-01-10'
    group by customer_id
) o
JOIN customers c
  ON o.customer_id = c.customer_id;

```

It is efficient because 
1. Data is filtered before joining
2. Data is aggregated before joining